# Lekcja 2.1 — eksperymenty: drzewo przeszukiwania a przestrzeń stanów

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz uruchomić notebook przed skończeniem zadania.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.search import reference as algorithms

Funkcja `levels(problem, depth)` buduje drzewo przeszukiwania poziom po poziomie funkcją `expand`, bez zapamiętywania stanów. Funkcja `growth(problem, depth)` zwraca dla każdej głębokości liczbę węzłów drzewa (z `count_tree_nodes`), liczbę różnych stanów na tej głębokości i liczbę wszystkich stanów osiągniętych do tej głębokości włącznie. Funkcja `show` wypisuje te liczby jako tabelę, a `plot_growth` rysuje je na wykresie.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search import Node
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"
SERIES = ((1, "#2a78d6", "tree nodes"), (2, "#eb6834", "distinct states"))


def levels(problem, depth):
    """The search tree level by level (lists of nodes), built with expand."""
    tree = [[Node(problem.initial)]]
    for _ in range(depth):
        tree.append([child for node in tree[-1] for child in algorithms.expand(problem, node)])
    return tree


def growth(problem, depth):
    """Rows of (depth, tree nodes, states at that depth, states reached so far)."""
    counts = list(algorithms.count_tree_nodes(problem, depth))
    rows, reached = [], set()
    for d, level in enumerate(levels(problem, depth)):
        if d >= len(counts) or counts[d] != len(level):
            print(f"warning: count_tree_nodes and expand disagree at depth {d} - check task.py")
        states = {node.state for node in level}
        reached |= states
        rows.append((d, counts[d] if d < len(counts) else len(level), len(states), len(reached)))
    return rows


def show(rows):
    print(f"{'depth':>5} {'tree nodes':>11} {'states':>7} {'reached':>8}")
    for d, nodes, states, reached in rows:
        print(f"{d:>5} {nodes:>11,} {states:>7} {reached:>8}")


def plot_growth(rows, title, ax):
    """Tree nodes and distinct states per depth, on a logarithmic scale."""
    depths = [row[0] for row in rows]
    for column, color, label in SERIES:
        values = [row[column] for row in rows]
        ax.plot(
            depths,
            values,
            color=color,
            linewidth=2,
            marker="o",
            markersize=8,
            markeredgecolor="#fcfcfb",
            markeredgewidth=1.5,
            label=label,
        )
        ax.annotate(
            f"{values[-1]:,}",
            (depths[-1], values[-1]),
            textcoords="offset points",
            xytext=(8, 0),
            va="center",
            color="#0b0b0b",
        )
    ax.set_yscale("log")
    ax.set_title(title, loc="left", color="#0b0b0b")
    ax.set_xlabel("depth", color="#52514e")
    ax.set_ylabel("count (log scale)", color="#52514e")
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")
    ax.margins(x=0.12)
    ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")

## Eksperyment 1: mapa z alejkami

Robot `R` stoi w górnym korytarzu. Budujemy drzewo przeszukiwania do głębokości 12, czyli wszystkie trasy o długości do 12 kroków.

In [ ]:
aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
print(render_text(aisles))

**Twoje przewidywanie.** Ile węzłów drzewa leży na głębokości 12? Ile różnych pól jest wśród nich? Wystarczy rząd wielkości.

*Moje przewidywanie:* …

In [ ]:
rows = growth(RouteProblem(aisles), 12)
show(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_growth(rows, "Search tree and state space, aisles.txt", ax)
fig.tight_layout()

**Do przemyślenia.** Porównaj wynik ze swoim przewidywaniem. Wykres ma skalę logarytmiczną: wykładniczy wzrost wygląda na nim jak linia prosta. Ile razy mniej więcej rośnie liczba węzłów drzewa z każdym poziomem? Co z tego wynika dla algorytmu, który nie zapamiętuje odwiedzonych stanów?

## Eksperyment 2: gdzie robot może być po 11, a gdzie po 12 krokach

Na obu rysunkach niebieskie pola to stany węzłów drzewa na danej głębokości.

**Twoje przewidywanie.** Czy pola osiągalne dokładnie w 11 krokach i dokładnie w 12 krokach mogą się pokrywać? Dlaczego liczba różnych stanów w tabeli raz rośnie, a raz maleje?

*Moje przewidywanie:* …

In [ ]:
tree = levels(RouteProblem(aisles), 12)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, depth in zip(axes, (11, 12)):
    cells = {node.state for node in tree[depth]}
    plot_map(
        aisles,
        expanded=cells,
        title=f"states at depth {depth}",
        ax=ax,
        expanded_label="state at this depth",
    )
fig.tight_layout()

**Do przemyślenia.** Pomaluj w myślach mapę jak szachownicę. Na jakie pola prowadzi każdy ruch? Co z tego wynika dla stanów na głębokości parzystej i nieparzystej? Kolumna `reached` w tabeli rośnie za to cały czas: ilu z tych stanów potrzebuje algorytm przeszukiwania grafu?

## Eksperyment 3: otwarta podłoga

Na mapie `open_floor` prawie nie ma przeszkód, więc z większości pól robot może jechać w czterech kierunkach. Ze względu na czas liczymy tylko do głębokości 8.

**Twoje przewidywanie.** Ile węzłów drzewa będzie na głębokości 8, w porównaniu z mapą z alejkami? A ile różnych stanów?

*Moje przewidywanie:* …

In [ ]:
open_floor = WarehouseMap.from_file(MAPS / "open_floor.txt")
open_rows = growth(RouteProblem(open_floor), 8)
show(open_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_growth(rows[:9], "aisles.txt", axes[0])
plot_growth(open_rows, "open_floor.txt", axes[1])
fig.tight_layout()

**Do przemyślenia.** Jak współczynnik rozgałęzienia wpływa na nachylenie niebieskiej linii? Ile razy więcej jest na otwartej podłodze węzłów drzewa, a ile razy więcej różnych stanów? Co przede wszystkim mnoży większe rozgałęzienie? Ile rozwinięć wykonałby BFS z lekcji 2.2, żeby zbadać wszystkie pola do głębokości 8?

## Podsumowanie

Wróć do pytań kontrolnych w lekcji. Po własnej próbie możesz porównać swoje wyniki i wnioski z odpowiedziami w `solutions/02_search/01_state_space/answers.pl.md`.